In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo

Because the official CrowdWater data downloaded via the Spotteron API only provides hydrological variables with encoded names (e.g. "fld_05_00000286"), it is necessary to decode them in order to understand what was actually measured in the CrowdWater observations.

Therefore, in the following, I create Lookup tables for all the variables. This is done by comparing the data to data downloaded from the Spotteron website, which has more understandable variable names (e.g. "PP_AMOUNT" (i.e. "Amount of Plastic Pieces)), and by comparing it with options in the CrowdWater app.

In [2]:
Category_LUT = pd.DataFrame({
    "CategoryID": [468,469,470,1919,3203,3204,9680],
    "Categorynr": [1,2,3,4,5,6,7],
    "CategoryInput": ["temporary stream","soil moisture","virtual scale","plastic pollution","physical scale","stream type","standing water type"]
})

WL_LUT = pd.DataFrame({
    "WLID": list(range(492, 506)),
    "WLInput": list(range(-6, 8))
})

SM_LUT = pd.DataFrame({
    "SMID": list(range(477, 485)),
    "SMnr": list(range(1, 9)),
    "SMInput": [
        "dry","gradually damp","gradually wet","immediately wet",
        "muddy","welling","submerged","rain / snow"
    ]
})

TS_LUT = pd.DataFrame({
    "TSID": list(range(471, 477)),
    "TSnr": [1,2,3,5,4,6],
    "TSInput": [
        "dry streambed","damp / wet streambed","isolated pools",
        "standing water","trickling water","flowing"
    ]
})

PP_LUT = pd.DataFrame({
    "PPID": list(range(1926, 1934)),
    "PPnr": list(range(1, 9)),
    "PPInput": [
        "no plastic","1-2 pieces","3-5 pieces","6-10 pieces",
        "11-20 pieces","21-100 pieces","100+ pieces","covered entirely"
    ]
})

Mat_LUT = pd.DataFrame({
    "MatID": [485,486,487,488,489,747,1288],
    "Matnr": list(range(1, 8)),
    "MatInput": [
        "sand","gravel","cobble","boulders",
        "bedrock","mud","concrete"
    ]
})

FlowVel_LUT = pd.DataFrame({
    "FlowVelID": [490,491],
    "FlowVelnr": [1,2],
    "FlowVelInput": [
        "direct","poo-stick"
    ]
})

PLoc_LUT = pd.DataFrame({
    "PLocID": [1920,1921],
    "PLocnr": [1,2],
    "PLocInput": [
        "Plastic in stream","Plastic on shore"
    ]
})

PWid_LUT = pd.DataFrame({
    "PWidID": [1922,1923,1924,1925],
    "PWidnr": [1,2,3,4],
    "PWidInput": [
        "all","1/2","1/3","1/4"
    ]
})

PET_LUT = pd.DataFrame({
    "PETID": [1934,1935,1936,1937,1938,1939],
    "PETnr": [1,2,3,4,5,6],
    "PETInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

POSoft_LUT = pd.DataFrame({
    "POSoftID": [1940,1941,1942,1943,1944,1945],
    "POSoftnr": [1,2,3,4,5,6],
    "POSoftInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

POHard_LUT = pd.DataFrame({
    "POHardID": [1946,1947,1948,1949,1950,1951],
    "POHardnr": [1,2,3,4,5,6],
    "POHardInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

PS_LUT = pd.DataFrame({
    "PSID": [1952,1953,1954,1955,1956,1957],
    "PSnr": [1,2,3,4,5,6],
    "PSInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

PSE_LUT = pd.DataFrame({
    "PSEID": [1958,1959,1960,1961,1962,1963],
    "PSEnr": [1,2,3,4,5,6],
    "PSEInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

PMultilayer_LUT = pd.DataFrame({
    "PMultilayerID": [1964,1965,1966,1967,1968,1969],
    "PMultilayernr": [1,2,3,4,5,6],
    "PMultilayerInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

POther_LUT = pd.DataFrame({
    "POtherID": [1970,1971,1972,1973,1974,1975],
    "POthernr": [1,2,3,4,5,6],
    "POtherInput": [
        "00%","20%","40%","60%","80%","100%"
    ]
})

PShoreDist_LUT = pd.DataFrame({
    "PShoreDistID": [1976,1977,1978,1979],
    "PShoreDistnr": [1,2,3,4],
    "PShoreDistInput": [
        "1 m","2 m","5 m","10 m"
    ]
})

PRemoved_LUT = pd.DataFrame({
    "PRemovedID": [1],
    "PRemovednr": [1],
    "PRemovedInput": [
        "Yes"
    ]
})

PRiverStag_LUT = pd.DataFrame({
    "PRiverStagID": [1],
    "PRiverStagnr": [1],
    "PRiverStagInput": [
        "Yes"
    ]
})

WaterLVLPhysicalUnit_LUT = pd.DataFrame({
    "WaterLVLPhysicalUnitID": [3205,3206,3207],
    "WaterLVLPhysicalUnitnr": [1,2,3],
    "WaterLVLPhysicalUnitInput": [
        "cm","inch","other_unit"
    ]
})

Streamtype_LUT = pd.DataFrame({
    "StreamtypeID": [3208,3209,3210,3211],
    "Streamtypenr": [1,2,3,4],
    "StreamtypeInput": [
        "Rinnsal","Bach","Fluss","Strom"
    ]
})

Swimming_LUT = pd.DataFrame({
    "SwimmingID": [3212,3213],
    "Swimmingnr": [1,2],
    "SwimmingInput": [
        "No","Yes"
    ]
})

Drinking_LUT = pd.DataFrame({
    "DrinkingID": [3214,3215],
    "Drinkingnr": [1,2],
    "DrinkingInput": [
        "No","Yes"
    ]
})

Naturality_LUT = pd.DataFrame({
    "NaturalityID": [3216,3217],
    "Naturalitynr": [1,2],
    "NaturalityInput": [
        "Natural","Controlled"
    ]
})

StreamColor_LUT = pd.DataFrame({
    "StreamColorID": [3218,3219,3220,3221,3222,6438,6439,6440,6441],
    "StreamColornr": [1,2,3,4,5,6,7,8,9],
    "StreamColorInput": [
        "Clear","Green","Brown","Gray","Yellow","Reddish","Blue","Gray","Beige"
    ]
})

Stream_Ground_Visibility_LUT = pd.DataFrame({
    "Stream_Ground_VisibilityID": [3223,3224,6755],
    "Stream_Ground_Visibilitynr": [1,2,3],
    "Stream_Ground_VisibilityInput": [
        "Yes","No","Vaguely"
    ]
})

Stream_Animals_LUT = pd.DataFrame({
    "Stream_AnimalsID": [3225,3226],
    "Stream_Animalsnr": [1,2],
    "Stream_AnimalsInput": [
        "Yes","No"
    ]
})

Stream_sometimes_dry_LUT = pd.DataFrame({
    "Stream_sometimes_dryID": [1],
    "Stream_sometimes_drynr": [1],
    "Stream_sometimes_dryInput": [
        "Yes"
    ]
})

TS_snow_ice_LUT = pd.DataFrame({
    "TS_snow_iceID": [1],
    "TS_snow_icenr": [1],
    "TS_snow_iceInput": [
        "Yes"
    ]
})

Waterquality_LUT = pd.DataFrame({
    "WaterqualityID": [6430,6431,6432,6433,6434],
    "Waterqualitynr": [1,2,3,4,5],
    "WaterqualityInput": [
        "Excellent","Good","Moderate","Poor","Very Poor"
    ]
})

Waterclarity_LUT = pd.DataFrame({
    "WaterclarityID": [6435,6436,6437],
    "Waterclaritynr": [1,2,3],
    "WaterclarityInput": [
        "Clear","Slightly Murky","Very Murky"
    ]
})

Stream_Vegetation_LUT = pd.DataFrame({
    "Stream_VegetationID": [6442,6443],
    "Stream_Vegetationnr": [1,2],
    "Stream_VegetationInput": [
        "Yes","No"
    ]
})

Stream_Foam_LUT = pd.DataFrame({
    "Stream_FoamID": [6444,6445,6446],
    "Stream_Foamnr": [1,2,3],
    "Stream_FoamInput": [
        "No","Yes, a little","Yes, a lot"
    ]
})

Stream_Algae_LUT = pd.DataFrame({
    "Stream_AlgaeID": [6529,6535,6536],
    "Stream_Algaenr": [1,2,3],
    "Stream_AlgaeInput": [
        "Clear Water","Growth visible","Large accumulation"
    ]
})

Stream_Odor_LUT = pd.DataFrame({
    "Stream_OdorID": [6537,6538],
    "Stream_Odornr": [1,2],
    "Stream_OdorInput": [
        "Yes","No"
    ]
})

Stream_Odor_Type_LUT = pd.DataFrame({
    "Stream_Odor_TypeID": [6539,6540,6541,6542,6543,6544],
    "Stream_Odor_Typenr": [1,2,3,4,5,6],
    "Stream_Odor_TypeInput": [
        "Rotten Eggs","Fishy","Oily","Trash","Sewage","Other"
    ]
})

Stream_Litter_LUT = pd.DataFrame({
    "Stream_LitterID": [6545,6546,6547],
    "Stream_Litternr": [1,2,3],
    "Stream_LitterInput": [
        "No","Some","Abundant"
    ]
})

Stream_Flow_Alteration_LUT = pd.DataFrame({
    "Stream_Flow_AlterationID": [6548,6549,6550],
    "Stream_Flow_Alterationnr": [1,2,3],
    "Stream_Flow_AlterationInput": [
        "No withdrawals, discharge or diversions from the water","Stormwater/Industrial/household discharge into the water","Temporary or constant withdrawals from the water"
    ]
})

Stream_typical_Color_LUT = pd.DataFrame({
    "Stream_typical_ColorID": [6756,6757,6758],
    "Stream_typical_Colornr": [1,2,3],
    "Stream_typical_ColorInput": [
        "Yes","No","I don't know"
    ]
})

Stream_Drainage_Basin_LUT = pd.DataFrame({
    "Stream_Drainage_BasinID": [31741,35694],
    "Stream_Drainage_Basinnr": [1,2],
    "Stream_Drainage_BasinInput": [
        "Lippe","Emscher"
    ]
})

Watertype_LUT = pd.DataFrame({
    "WatertypeID": [9801,9802],
    "Watertypenr": [1,2],
    "WatertypeInput": [
        "River and Land","Lake"
    ]
})

Lake_Usage_LUT = pd.DataFrame({
    "Lake_UsageID": [9803,9804,9805,9806,9807,9808],
    "Lake_Usagenr": [1,2,3,4,5,6],
    "Lake_UsageInput": [
        "Bathing","Dog Bathing","Water Sports","Fishing","Activities on the Shore","No visible Usage"
    ]
})

Lake_Access_LUT = pd.DataFrame({
    "Lake_AccessID": [9809,9810],
    "Lake_Accessnr": [1,2],
    "Lake_AccessInput": [
        "Yes", "No"
    ]
})

Lake_Shore_State_LUT = pd.DataFrame({
    "Lake_Shore_StateID": [9811,9812,9813],
    "Lake_Shore_Statenr": [1,2,3],
    "Lake_Shore_StateInput": [
        "Natural", "Partly Built-In","Built-In"
    ]
})

Lake_Swimming_LUT = pd.DataFrame({
    "Lake_SwimmingID": [9814,9815],
    "Lake_Swimmingnr": [1,2],
    "Lake_SwimmingInput": [
        "Yes", "No"
    ]
})

Lake_Transparency_LUT = pd.DataFrame({
    "Lake_TransparencyID": [9816,9817,9818,9819],
    "Lake_Transparencynr": [1,2,3,4],
    "Lake_TransparencyInput": [
        "a few cm","20-50cm","50cm-1m","deeper than 1m"
    ]
})

Lake_Color_LUT = pd.DataFrame({
    "Lake_ColorID": [9820,9821,9822,9823,9824,9825,9826],
    "Lake_Colornr": [1,2,3,4,5,6,7],
    "Lake_ColorInput": [
        "Clear","Green","Beige","Brown","Gray","Reddish","Other"
    ]
})

Lake_Odor_LUT = pd.DataFrame({
    "Lake_OdorID": [9827,9828,9829,9830,9831,9832],
    "Lake_Odornr": [1,2,3,4,5,6],
    "Lake_OdorInput": [
        "No Odor","Rotten Eggs","Musty","Fishy","Sewage","Other Odor"
    ]
})

Lake_Shore_Vegetation_LUT = pd.DataFrame({
    "Lake_Shore_VegetationID": [9833,9834,9835,9836,9837],
    "Lake_Shore_Vegetationnr": [1,2,3,4,5],
    "Lake_Shore_VegetationInput": [
        "Reed","Trees","Lawn","Other","No Vegetation"
    ]
})

Lake_Underwater_Vegetation_LUT = pd.DataFrame({
    "Lake_Underwater_VegetationID": [9838,9839],
    "Lake_Underwater_Vegetationnr": [1,2],
    "Lake_Underwater_VegetationInput": [
        "Yes","No"
    ]
})

Lake_Floating_Leaveplants_LUT = pd.DataFrame({
    "Lake_Floating_LeaveplantsID": [9840,9841],
    "Lake_Floating_Leaveplantsnr": [1,2],
    "Lake_Floating_LeaveplantsInput": [
        "Yes","No"
    ]
})

Lake_Duckweed_LUT = pd.DataFrame({
    "Lake_DuckweedID": [9842,9843,9844,9845,9846],
    "Lake_Duckweednr": [1,2,3,4,5],
    "Lake_DuckweedInput": [
        "Zero","ca 1/4","ca 1/2","ca 3/4","Completely"
    ]
})

Lake_Mussels_LUT = pd.DataFrame({
    "Lake_MusselsID": [9850,9851,9852,9853],
    "Lake_Musselsnr": [1,2,3,4],
    "Lake_MusselsInput": [
        "Large Mussels","Dreissenidae","Other Mussels","No Mussels"
    ]
})

Lake_Deadwood_LUT = pd.DataFrame({
    "Lake_DeadwoodID": [9854,9855,9856],
    "Lake_Deadwoodnr": [1,2,3],
    "Lake_DeadwoodInput": [
        "Yes, a lot","Yes, some","No"
    ]
})

Lake_Animals_LUT = pd.DataFrame({
    "Lake_AnimalsID": [9857,9858,9859,9860,9861],
    "Lake_Animalsnr": [1,2,3,4,5],
    "Lake_AnimalsInput": [
        "Fish","Amphibians","Dragonflies","Waterfowl","No visible Animals"
    ]
})

Lake_Waterlevel_Changes_LUT = pd.DataFrame({
    "Lake_Waterlevel_ChangesID": [9862,9863,9864,9865],
    "Lake_Waterlevel_Changesnr": [1,2,3,4],
    "Lake_Waterlevel_ChangesInput": [
        "Yes, large","Yes, small","No","Unknown"
    ]
})

Lake_Dry_LUT = pd.DataFrame({
    "Lake_DryID": [9866,9867,9868],
    "Lake_Drynr": [1,2,3],
    "Lake_DryInput": [
        "Yes","No","Unknown"
    ]
})

Here, I download the CrowdWater data via the Spotteron API and use the Lookup tables created above to change the names of the variables and the observations values so that humans can understand and read the data.

In [3]:
def download_CW_data():
    base_url = "https://www.spotteron.com/api/v2/spots"
    params = {
        "filter[topic_id]": 7,
        "filter[created_at__gt]": "2016-01-01 14:30:00",
        "limit": 100,
        "page": 1,
        "order[]": "created_at asc"
    }
    all_rows = []
    counter = 1
    while True:

        r = requests.get(base_url, params=params)
        data = r.json()["data"]

        if not data:
            break

        rows = []
        for item in data:
            row = {"id": item["id"]}
            row.update(item["attributes"])
            rows.append(row)

        df = pd.DataFrame(rows)
        all_rows.append(df)

        print(f"Page {counter} downloaded")

        total_rows = sum(len(x) for x in all_rows)
        #if total_rows >= limit_rows:
        #    print(f"Reached limit of {limit_rows} rows.")
        #    break

        # nächste Seite vorbereiten
        last_date = df["created_at"].iloc[-1]
        params["filter[created_at__gt]"] = last_date
        counter += 1

        time.sleep(1)

    t_data = pd.concat(all_rows, ignore_index=True)

    t_data["Category"] = t_data["category"].map(
        dict(zip(Category_LUT.CategoryID, Category_LUT.CategoryInput))
    )

    t_data["Category_Nr"] = t_data["category"].map(
        dict(zip(Category_LUT.CategoryID, Category_LUT.Categorynr))
    )
        
    t_data["Waterlevel_Virtual"] = t_data["fld_05_00000066"].map(
        dict(zip(WL_LUT.WLID, WL_LUT.WLInput))
    )

    t_data["SoilMoisture"] = t_data["fld_05_00000052"].map(
        dict(zip(SM_LUT.SMID, SM_LUT.SMInput))
    )

    t_data["SoilMoisture_Nr"] = t_data["fld_05_00000052"].map(
        dict(zip(SM_LUT.SMID, SM_LUT.SMnr))
    )

    t_data["TempStream"] = t_data["fld_05_00000051"].map(
        dict(zip(TS_LUT.TSID, TS_LUT.TSInput))
    )

    t_data["TempStream_Nr"] = t_data["fld_05_00000051"].map(
        dict(zip(TS_LUT.TSID, TS_LUT.TSnr))
    )

    t_data["Plastic_Amount"] = t_data["fld_05_00000286"].map(
        dict(zip(PP_LUT.PPID, PP_LUT.PPInput))
    )

    t_data["Plastic_Amount_Nr"] = t_data["fld_05_00000286"].map(
        dict(zip(PP_LUT.PPID, PP_LUT.PPnr))
    )

    t_data["Stream_Width"] = t_data["fld_11_00000054"]

    t_data["Stream_Depth"] = t_data["fld_11_00000055"]

    t_data["Streambed_Material"] = t_data["fld_03_00000056"].map(
        dict(zip(Mat_LUT.MatID, Mat_LUT.MatInput))
    )

    t_data["Streambed_Material_Nr"] = t_data["fld_03_00000056"].map(
        dict(zip(Mat_LUT.MatID, Mat_LUT.Matnr))
    )

    t_data["FlowVelocity_Method"] = t_data["fld_05_00000057"].map(
        dict(zip(FlowVel_LUT.FlowVelID, FlowVel_LUT.FlowVelInput))
    )

    t_data["FlowVelocity_Method_Nr"] = t_data["fld_05_00000057"].map(
        dict(zip(FlowVel_LUT.FlowVelID, FlowVel_LUT.FlowVelnr))
    )

    t_data["FlowVelocity_direct_vel"] = t_data["fld_11_00000059"]

    t_data["FlowVelocity_PS_dist"] = t_data["fld_11_00000060"]

    t_data["FlowVelocity_PS_time1"] = t_data["fld_11_00000061"]

    t_data["FlowVelocity_PS_time2"] = t_data["fld_11_00000063"]

    t_data["FlowVelocity_PS_time3"] = t_data["fld_11_00000065"]

    t_data["Plastic_ObservationTime"] = t_data["fld_11_00000283"]

    t_data["Plastic_Location"] = t_data["fld_05_00000284"].map(
        dict(zip(PLoc_LUT.PLocID, PLoc_LUT.PLocInput))
    )

    t_data["Plastic_Location_Nr"] = t_data["fld_05_00000284"].map(
        dict(zip(PLoc_LUT.PLocID, PLoc_LUT.PLocnr))
    )

    t_data["Plastic_RiverWidth"] = t_data["fld_05_00000285"].map(
        dict(zip(PWid_LUT.PWidID, PWid_LUT.PWidInput))
    )

    t_data["Plastic_RiverWidth_Nr"] = t_data["fld_05_00000285"].map(
        dict(zip(PWid_LUT.PWidID, PWid_LUT.PWidnr))
    )

    t_data["Plastic_PET"] = t_data["fld_05_00000288"].map(
        dict(zip(PET_LUT.PETID, PET_LUT.PETInput))
    )

    t_data["Plastic_PET_Nr"] = t_data["fld_05_00000288"].map(
        dict(zip(PET_LUT.PETID, PET_LUT.PETnr))
    )

    t_data["Plastic_POSoft"] = t_data["fld_05_00000289"].map(
        dict(zip(POSoft_LUT.POSoftID, POSoft_LUT.POSoftInput))
    )

    t_data["Plastic_POSoft_Nr"] = t_data["fld_05_00000289"].map(
        dict(zip(POSoft_LUT.POSoftID, POSoft_LUT.POSoftnr))
    )

    t_data["Plastic_POHard"] = t_data["fld_05_00000291"].map(
        dict(zip(POHard_LUT.POHardID, POHard_LUT.POHardInput))
    )

    t_data["Plastic_POHard_Nr"] = t_data["fld_05_00000291"].map(
        dict(zip(POHard_LUT.POHardID, POHard_LUT.POHardnr))
    )

    t_data["Plastic_PS"] = t_data["fld_05_00000292"].map(
        dict(zip(PS_LUT.PSID, PS_LUT.PSInput))
    )

    t_data["Plastic_PS_Nr"] = t_data["fld_05_00000292"].map(
        dict(zip(PS_LUT.PSID, PS_LUT.PSnr))
    )

    t_data["Plastic_PSE"] = t_data["fld_05_00000293"].map(
        dict(zip(PSE_LUT.PSEID, PSE_LUT.PSEInput))
    )

    t_data["Plastic_PSE_Nr"] = t_data["fld_05_00000293"].map(
        dict(zip(PSE_LUT.PSEID, PSE_LUT.PSEnr))
    )

    t_data["Plastic_PMultilayer"] = t_data["fld_05_00000294"].map(
        dict(zip(PMultilayer_LUT.PMultilayerID, PMultilayer_LUT.PMultilayerInput))
    )

    t_data["Plastic_PMultilayer_Nr"] = t_data["fld_05_00000294"].map(
        dict(zip(PMultilayer_LUT.PMultilayerID, PMultilayer_LUT.PMultilayernr))
    )

    t_data["Plastic_POther"] = t_data["fld_05_00000295"].map(
        dict(zip(POther_LUT.POtherID, POther_LUT.POtherInput))
    )

    t_data["Plastic_POther_Nr"] = t_data["fld_05_00000295"].map(
        dict(zip(POther_LUT.POtherID, POther_LUT.POthernr))
    )

    t_data["Plastic_Shore_Plotsize"] = t_data["fld_05_00000296"].map(
        dict(zip(PShoreDist_LUT.PShoreDistID, PShoreDist_LUT.PShoreDistInput))
    )

    t_data["Plastic_Shore_Plotsize_Nr"] = t_data["fld_05_00000296"].map(
        dict(zip(PShoreDist_LUT.PShoreDistID, PShoreDist_LUT.PShoreDistnr))
    )

    t_data["Plastic_Removed"] = t_data["fld_07_00000411"].map(
        dict(zip(PRemoved_LUT.PRemovedID, PRemoved_LUT.PRemovedInput))
    )

    t_data["Plastic_River_Stagnant"] = t_data["fld_07_00000412"].map(
        dict(zip(PRiverStag_LUT.PRiverStagID, PRiverStag_LUT.PRiverStagInput))
    )

    t_data["Waterlevel_Physical_Unit"] = t_data["fld_05_00000619"].map(
        dict(zip(WaterLVLPhysicalUnit_LUT.WaterLVLPhysicalUnitID, WaterLVLPhysicalUnit_LUT.WaterLVLPhysicalUnitInput))
    )

    t_data["Waterlevel_Physical_Unit_Nr"] = t_data["fld_05_00000619"].map(
        dict(zip(WaterLVLPhysicalUnit_LUT.WaterLVLPhysicalUnitID, WaterLVLPhysicalUnit_LUT.WaterLVLPhysicalUnitnr))
    )

    t_data["Waterlevel_Physical"] = t_data["fld_11_00000620"]

    t_data["Streamtype"] = t_data["fld_05_00000621"].map(
        dict(zip(Streamtype_LUT.StreamtypeID, Streamtype_LUT.StreamtypeInput))
    )

    t_data["Streamtype_Nr"] = t_data["fld_05_00000621"].map(
        dict(zip(Streamtype_LUT.StreamtypeID, Streamtype_LUT.Streamtypenr))
    )

    t_data["Swimming_Quality"] = t_data["fld_05_00000622"].map(
        dict(zip(Swimming_LUT.SwimmingID, Swimming_LUT.SwimmingInput))
    )

    t_data["Drinking_Quality"] = t_data["fld_05_00000623"].map(
        dict(zip(Drinking_LUT.DrinkingID, Drinking_LUT.DrinkingInput))
    )

    t_data["Naturality"] = t_data["fld_05_00000624"].map(
        dict(zip(Naturality_LUT.NaturalityID, Naturality_LUT.NaturalityInput))
    )

    t_data["Naturality_Nr"] = t_data["fld_05_00000624"].map(
        dict(zip(Naturality_LUT.NaturalityID, Naturality_LUT.Naturalitynr))
    )

    t_data["StreamColor"] = t_data["fld_03_00000625"].map(
        dict(zip(StreamColor_LUT.StreamColorID, StreamColor_LUT.StreamColorInput))
    )

    t_data["StreamColor_Nr"] = t_data["fld_03_00000625"].map(
        dict(zip(StreamColor_LUT.StreamColorID, StreamColor_LUT.StreamColornr))
    )

    t_data["Stream_Ground_Visibility"] = t_data["fld_05_00000626"].map(
        dict(zip(Stream_Ground_Visibility_LUT.Stream_Ground_VisibilityID, Stream_Ground_Visibility_LUT.Stream_Ground_VisibilityInput))
    )

    t_data["Stream_Ground_Visibility_Nr"] = t_data["fld_05_00000626"].map(
        dict(zip(Stream_Ground_Visibility_LUT.Stream_Ground_VisibilityID, Stream_Ground_Visibility_LUT.Stream_Ground_Visibilitynr))
    )

    t_data["Stream_Animals"] = t_data["fld_05_00000627"].map(
        dict(zip(Stream_Animals_LUT.Stream_AnimalsID, Stream_Animals_LUT.Stream_AnimalsInput))
    )

    t_data["Stream_Pollution_Reason"] = t_data["fld_01_00000628"]

    t_data["Stream_sometimes_dry"] = t_data["fld_07_00000629"].map(
        dict(zip(Stream_sometimes_dry_LUT.Stream_sometimes_dryID, Stream_sometimes_dry_LUT.Stream_sometimes_dryInput))
    )

    t_data["Stream_Name"] = t_data["fld_01_00000630"]

    t_data["TempStream_snow_ice"] = t_data["fld_07_00000903"].map(
        dict(zip(TS_snow_ice_LUT.TS_snow_iceID, TS_snow_ice_LUT.TS_snow_iceInput))
    )

    t_data["Stream_Waterquality"] = t_data["fld_03_00001215"].map(
        dict(zip(Waterquality_LUT.WaterqualityID, Waterquality_LUT.WaterqualityInput))
    )

    t_data["Stream_Waterquality_Nr"] = t_data["fld_03_00001215"].map(
        dict(zip(Waterquality_LUT.WaterqualityID, Waterquality_LUT.Waterqualitynr))
    )

    t_data["Stream_Waterclarity"] = t_data["fld_05_00001216"].map(
        dict(zip(Waterclarity_LUT.WaterclarityID, Waterclarity_LUT.WaterclarityInput))
    )

    t_data["Stream_Waterclarity_Nr"] = t_data["fld_05_00001216"].map(
        dict(zip(Waterclarity_LUT.WaterclarityID, Waterclarity_LUT.Waterclaritynr))
    )

    t_data["StreamColor_other"] = t_data["fld_01_00001217"]

    t_data["Stream_Vegetation"] = t_data["fld_05_00001218"].map(
        dict(zip(Stream_Vegetation_LUT.Stream_VegetationID, Stream_Vegetation_LUT.Stream_VegetationInput))
    )

    t_data["Stream_Foam"] = t_data["fld_05_00001219"].map(
        dict(zip(Stream_Foam_LUT.Stream_FoamID, Stream_Foam_LUT.Stream_FoamInput))
    )

    t_data["Stream_Foam_Nr"] = t_data["fld_05_00001219"].map(
        dict(zip(Stream_Foam_LUT.Stream_FoamID, Stream_Foam_LUT.Stream_Foamnr))
    )

    t_data["Stream_Algae"] = t_data["fld_03_00001244"].map(
        dict(zip(Stream_Algae_LUT.Stream_AlgaeID, Stream_Algae_LUT.Stream_AlgaeInput))
    )

    t_data["Stream_Algae_Nr"] = t_data["fld_03_00001244"].map(
        dict(zip(Stream_Algae_LUT.Stream_AlgaeID, Stream_Algae_LUT.Stream_Algaenr))
    )

    t_data["Stream_Odor"] = t_data["fld_05_00001247"].map(
        dict(zip(Stream_Odor_LUT.Stream_OdorID, Stream_Odor_LUT.Stream_OdorInput))
    )

    t_data["Stream_Odor_Type"] = t_data["fld_05_00001248"].map(
        dict(zip(Stream_Odor_Type_LUT.Stream_Odor_TypeID, Stream_Odor_Type_LUT.Stream_Odor_TypeInput))
    )

    t_data["Stream_Odor_Type_Nr"] = t_data["fld_05_00001248"].map(
        dict(zip(Stream_Odor_Type_LUT.Stream_Odor_TypeID, Stream_Odor_Type_LUT.Stream_Odor_Typenr))
    )

    t_data["Stream_Odor_Type_other"] = t_data["fld_01_00001249"]

    t_data["Stream_Litter"] = t_data["fld_05_00001250"].map(
        dict(zip(Stream_Litter_LUT.Stream_LitterID, Stream_Litter_LUT.Stream_LitterInput))
    )

    t_data["Stream_Litter_Nr"] = t_data["fld_05_00001250"].map(
        dict(zip(Stream_Litter_LUT.Stream_LitterID, Stream_Litter_LUT.Stream_Litternr))
    )

    t_data["Stream_Flow_Alteration"] = t_data["fld_03_00001252"].map(
        dict(zip(Stream_Flow_Alteration_LUT.Stream_Flow_AlterationID, Stream_Flow_Alteration_LUT.Stream_Flow_AlterationInput))
    )

    t_data["Stream_Flow_Alteration_Nr"] = t_data["fld_03_00001252"].map(
        dict(zip(Stream_Flow_Alteration_LUT.Stream_Flow_AlterationID, Stream_Flow_Alteration_LUT.Stream_Flow_Alterationnr))
    )

    t_data["Stream_typical_Color"] = t_data["fld_05_00001333"].map(
        dict(zip(Stream_typical_Color_LUT.Stream_typical_ColorID, Stream_typical_Color_LUT.Stream_typical_ColorInput))
    )

    t_data["Stream_typical_Color_Nr"] = t_data["fld_05_00001333"].map(
        dict(zip(Stream_typical_Color_LUT.Stream_typical_ColorID, Stream_typical_Color_LUT.Stream_typical_Colornr))
    )

    t_data["Stream_Drainage_Basin"] = t_data["fld_15_00001890"].map(
        dict(zip(Stream_Drainage_Basin_LUT.Stream_Drainage_BasinID, Stream_Drainage_Basin_LUT.Stream_Drainage_BasinInput))
    )

    t_data["Stream_Drainage_Basin_Nr"] = t_data["fld_15_00001890"].map(
        dict(zip(Stream_Drainage_Basin_LUT.Stream_Drainage_BasinID, Stream_Drainage_Basin_LUT.Stream_Drainage_Basinnr))
    )

    t_data["Watertype"] = t_data["fld_05_00002144"].map(
        dict(zip(Watertype_LUT.WatertypeID, Watertype_LUT.WatertypeInput))
    )

    t_data["Watertype_Nr"] = t_data["fld_05_00002144"].map(
        dict(zip(Watertype_LUT.WatertypeID, Watertype_LUT.Watertypenr))
    )

    Lake_Usage_map = dict(zip(
        Lake_Usage_LUT.Lake_UsageID,
        Lake_Usage_LUT.Lake_UsageInput
    ))

    Lake_Usage_Nr_map = dict(zip(
        Lake_Usage_LUT.Lake_UsageID,
        Lake_Usage_LUT.Lake_Usagenr
    ))
    
    def parse_list(x):
        if isinstance(x, str):
            return ast.literal_eval(x)
        return x
    
    def map_list(values, mapping):
        if not isinstance(values, list):
            return None
        return [mapping.get(v) for v in values]
    
    col = t_data["fld_04_00002145"].apply(parse_list)

    t_data["Lake_Usage"] = col.apply(lambda x: map_list(x, Lake_Usage_map))
    t_data["Lake_Usage_Nr"] = col.apply(lambda x: map_list(x, Lake_Usage_Nr_map))

    t_data["Lake_Access"] = t_data["fld_05_00002146"].map(
        dict(zip(Lake_Access_LUT.Lake_AccessID, Lake_Access_LUT.Lake_AccessInput))
    )

    t_data["Lake_Shore_State"] = t_data["fld_05_00002147"].map(
        dict(zip(Lake_Shore_State_LUT.Lake_Shore_StateID, Lake_Shore_State_LUT.Lake_Shore_StateInput))
    )

    t_data["Lake_Shore_State_Nr"] = t_data["fld_05_00002147"].map(
        dict(zip(Lake_Shore_State_LUT.Lake_Shore_StateID, Lake_Shore_State_LUT.Lake_Shore_Statenr))
    )

    t_data["Lake_Swimming"] = t_data["fld_05_00002148"].map(
        dict(zip(Lake_Swimming_LUT.Lake_SwimmingID, Lake_Swimming_LUT.Lake_SwimmingInput))
    )

    t_data["Lake_Transparency"] = t_data["fld_03_00002149"].map(
        dict(zip(Lake_Transparency_LUT.Lake_TransparencyID, Lake_Transparency_LUT.Lake_TransparencyInput))
    )

    t_data["Lake_Transparency_Nr"] = t_data["fld_03_00002149"].map(
        dict(zip(Lake_Transparency_LUT.Lake_TransparencyID, Lake_Transparency_LUT.Lake_Transparencynr))
    )

    t_data["Lake_Color"] = t_data["fld_03_00002150"].map(
        dict(zip(Lake_Color_LUT.Lake_ColorID, Lake_Color_LUT.Lake_ColorInput))
    )

    t_data["Lake_Color_Nr"] = t_data["fld_03_00002150"].map(
        dict(zip(Lake_Color_LUT.Lake_ColorID, Lake_Color_LUT.Lake_Colornr))
    )

    t_data["Lake_Odor"] = t_data["fld_03_00002151"].map(
        dict(zip(Lake_Odor_LUT.Lake_OdorID, Lake_Odor_LUT.Lake_OdorInput))
    )

    t_data["Lake_Odor_Nr"] = t_data["fld_03_00002151"].map(
        dict(zip(Lake_Odor_LUT.Lake_OdorID, Lake_Odor_LUT.Lake_Odornr))
    )

    Lake_Shore_Vegetation_map = dict(zip(
        Lake_Shore_Vegetation_LUT.Lake_Shore_VegetationID,
        Lake_Shore_Vegetation_LUT.Lake_Shore_VegetationInput
    ))

    Lake_Shore_Vegetation_Nr_map = dict(zip(
        Lake_Shore_Vegetation_LUT.Lake_Shore_VegetationID,
        Lake_Shore_Vegetation_LUT.Lake_Shore_Vegetationnr
    ))

    col2 = t_data["fld_04_00002152"].apply(parse_list)

    t_data["Lake_Shore_Vegetation"] = col2.apply(lambda x: map_list(x, Lake_Shore_Vegetation_map))
    t_data["Lake_Shore_Vegetation_Nr"] = col2.apply(lambda x: map_list(x, Lake_Shore_Vegetation_Nr_map))

    t_data["Lake_Underwater_Vegetation"] = t_data["fld_05_00002153"].map(
        dict(zip(Lake_Underwater_Vegetation_LUT.Lake_Underwater_VegetationID, Lake_Underwater_Vegetation_LUT.Lake_Underwater_VegetationInput))
    )

    t_data["Lake_Floating_Leaveplants"] = t_data["fld_05_00002154"].map(
        dict(zip(Lake_Floating_Leaveplants_LUT.Lake_Floating_LeaveplantsID, Lake_Floating_Leaveplants_LUT.Lake_Floating_LeaveplantsInput))
    )

    t_data["Lake_Duckweed"] = t_data["fld_03_00002156"].map(
        dict(zip(Lake_Duckweed_LUT.Lake_DuckweedID, Lake_Duckweed_LUT.Lake_DuckweedInput))
    )

    t_data["Lake_Duckweed_Nr"] = t_data["fld_03_00002156"].map(
        dict(zip(Lake_Duckweed_LUT.Lake_DuckweedID, Lake_Duckweed_LUT.Lake_Duckweednr))
    )

    t_data["Lake_Mussels"] = t_data["fld_05_00002158"].map(
        dict(zip(Lake_Mussels_LUT.Lake_MusselsID, Lake_Mussels_LUT.Lake_MusselsInput))
    )

    t_data["Lake_Mussels_Nr"] = t_data["fld_05_00002158"].map(
        dict(zip(Lake_Mussels_LUT.Lake_MusselsID, Lake_Mussels_LUT.Lake_Musselsnr))
    )

    t_data["Lake_Deadwood"] = t_data["fld_03_00002159"].map(
        dict(zip(Lake_Deadwood_LUT.Lake_DeadwoodID, Lake_Deadwood_LUT.Lake_DeadwoodInput))
    )

    t_data["Lake_Deadwood_Nr"] = t_data["fld_03_00002159"].map(
        dict(zip(Lake_Deadwood_LUT.Lake_DeadwoodID, Lake_Deadwood_LUT.Lake_Deadwoodnr))
    )

    Lake_Animals_map = dict(zip(
        Lake_Animals_LUT.Lake_AnimalsID,
        Lake_Animals_LUT.Lake_AnimalsInput
    ))

    Lake_Animals_Nr_map = dict(zip(
        Lake_Animals_LUT.Lake_AnimalsID,
        Lake_Animals_LUT.Lake_Animalsnr
    ))

    col3 = t_data["fld_04_00002160"].apply(parse_list)

    t_data["Lake_Animals"] = col3.apply(lambda x: map_list(x, Lake_Animals_map))
    t_data["Lake_Animals_Nr"] = col3.apply(lambda x: map_list(x, Lake_Animals_Nr_map))

    t_data["Lake_Waterlevel_Changes"] = t_data["fld_03_00002162"].map(
        dict(zip(Lake_Waterlevel_Changes_LUT.Lake_Waterlevel_ChangesID, Lake_Waterlevel_Changes_LUT.Lake_Waterlevel_ChangesInput))
    )

    t_data["Lake_Waterlevel_Changes_Nr"] = t_data["fld_03_00002162"].map(
        dict(zip(Lake_Waterlevel_Changes_LUT.Lake_Waterlevel_ChangesID, Lake_Waterlevel_Changes_LUT.Lake_Waterlevel_Changesnr))
    )

    t_data["Lake_Dry"] = t_data["fld_03_00002163"].map(
        dict(zip(Lake_Dry_LUT.Lake_DryID, Lake_Dry_LUT.Lake_DryInput))
    )

    t_data["Lake_Dry_Nr"] = t_data["fld_03_00002163"].map(
        dict(zip(Lake_Dry_LUT.Lake_DryID, Lake_Dry_LUT.Lake_Drynr))
    )

    t_data["Image_Gallery"] = t_data["fld_21_00002431"]

    t_data = t_data.loc[:, ~t_data.columns.str.startswith("fld_")]

    return t_data

Use the function

In [4]:
CWData_clean = download_CW_data()

CWData_clean.to_csv("../CWData_clean.csv", index=False)

Page 1 downloaded
Page 2 downloaded
Page 3 downloaded
Page 4 downloaded
Page 5 downloaded
Page 6 downloaded
Page 7 downloaded
Page 8 downloaded
Page 9 downloaded
Page 10 downloaded
Page 11 downloaded
Page 12 downloaded
Page 13 downloaded
Page 14 downloaded
Page 15 downloaded
Page 16 downloaded
Page 17 downloaded
Page 18 downloaded
Page 19 downloaded
Page 20 downloaded
Page 21 downloaded
Page 22 downloaded
Page 23 downloaded
Page 24 downloaded
Page 25 downloaded
Page 26 downloaded
Page 27 downloaded
Page 28 downloaded
Page 29 downloaded
Page 30 downloaded
Page 31 downloaded
Page 32 downloaded
Page 33 downloaded
Page 34 downloaded
Page 35 downloaded
Page 36 downloaded
Page 37 downloaded
Page 38 downloaded
Page 39 downloaded
Page 40 downloaded
Page 41 downloaded
Page 42 downloaded
Page 43 downloaded
Page 44 downloaded
Page 45 downloaded
Page 46 downloaded
Page 47 downloaded
Page 48 downloaded
Page 49 downloaded
Page 50 downloaded
Page 51 downloaded
Page 52 downloaded
Page 53 downloaded
Pa

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\2431139151.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  t_data["Stream_Width"] = t_data["fld_11_00000054"]
C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\2431139151.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  t_data["Stream_Depth"] = t_data["fld_11_00000055"]
C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\2431139151.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

Add created_at_local which is the local time of an observation since the other one is UTC time. I do not use spotted_at (which is already local time) because that would not make any sense. E.g. someone finding the app in 2020 can just create observations dating back to 2018, if I then use 2018 as the relevant date, that would falsify the spread.

In [5]:
df = pd.read_csv("../CWData_clean.csv")
df["created_at"] = pd.to_datetime(df["created_at"])

df = df.dropna(subset=["latitude", "longitude", "created_at"])
df = df[(df["latitude"] != 0) & (df["longitude"] != 0)] # remove 0/0 locations

tf = TimezoneFinder()

def get_tz_name(lat: float, lon: float) -> str | None:
    return tf.timezone_at(lat=lat, lng=lon)

# time zone column pre-calculated (look-up table)
df["tz_name"] = df.apply(lambda r: get_tz_name(r["latitude"], r["longitude"]), axis=1)

def utc_to_local(utc_timestamp: datetime, tz_name: str) -> datetime:
    if pd.isna(tz_name):
        return utc_timestamp  # fallback: keep UTC
    if utc_timestamp.tzinfo is None:
        utc_timestamp = utc_timestamp.replace(tzinfo=timezone.utc)
    return utc_timestamp.astimezone(zoneinfo.ZoneInfo(tz_name))

df["created_at_local"] = df.apply(
    lambda r: utc_to_local(r["created_at"], r["tz_name"]),
    axis=1
)

df.to_csv("../CWData_clean2.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\306096296.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: Lak

Stop after June 2026. I'm using created_at_local from now on

In [6]:
df = pd.read_csv("../CWData_clean2.csv")
df = df[df["created_at_local"] < "2026-07-01"]
df.to_csv("../CWData_clean3.csv",index=False) # new dataset with data up to and including April 2026 (but not further)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\1806971041.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: La

In the resulting dataframe, I add the country and the city of the observation location using the coordinates of the raw data.

In [9]:
df = pd.read_csv("../CWData_clean3.csv")
df = df.dropna(subset=["latitude","longitude"])

world = gpd.read_file("../Borders/ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp")[["ADM0_A3", "NAME", "geometry"]]

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

gdf = gpd.sjoin(gdf, world, how="left", predicate="within") # Country gets assigned to spots

gdf = gdf.rename(columns={ # rename some columns
    "ADM0_A3": "ISO_A3",
    "NAME": "Country"
})

# Cyprus has a few unnecessary regions + the spots in Northern Cyprus are actually in Southern Cyprus (marginally)
country_fix = {
    "Akrotiri": "Cyprus",
    "Cyprus U.N. Buffer Zone": "Cyprus",
    "Dhekelia": "Cyprus",
    "N. Cyprus": "Cyprus"
}

gdf["Country"] = gdf["Country"].replace(country_fix)

iso_fix = {name: "CYP" for name in country_fix} # the ISO_A3 code also has to be fixed for Cyprus
gdf["ISO_A3"] = gdf["ISO_A3"].replace(iso_fix)

# some spots near the coast have no country
world_proj = world.to_crs("EPSG:3857") # other projection for distance calculations

missing = gdf["Country"].isna()
gdf_missing = gdf[missing].copy()
gdf_missing = gdf_missing.drop(columns=["index_right"], errors="ignore")
gdf_missing_proj = gdf_missing.to_crs("EPSG:3857") # reproject
gdf_missing_proj = gpd.sjoin_nearest(gdf_missing_proj, world_proj[["ADM0_A3","NAME","geometry"]], how="left") # get the nearest country to the location
gdf_missing_proj = gdf_missing_proj[~gdf_missing_proj.index.duplicated(keep="first")] # if two countries are found that are the same distance away
gdf.loc[missing, "Country"] = gdf_missing_proj["NAME"] # transfer the names of the countries to the original dataframe
gdf.loc[missing, "ISO_A3"]  = gdf_missing_proj["ADM0_A3"]

# cities
cities = gpd.read_file("../Borders/ne_10m_populated_places/ne_10m_populated_places.shp")[["NAME", "geometry"]]
cities = cities.rename(columns={"NAME": "City"})
cities_proj = cities.to_crs("EPSG:3857") # reproject

gdf_proj = gdf.drop(columns=["index_right"], errors="ignore").to_crs("EPSG:3857")
gdf_proj = gpd.sjoin_nearest(gdf_proj, cities_proj, how="left")
gdf["City"] = gdf_proj["City"].values

gdf["created_at_local"] = pd.to_datetime(gdf["created_at_local"].str[:19]) # remove timezone info
gdf["year_month"] = gdf["created_at_local"].dt.to_period("M")

gdf.to_csv("../CWData_clean4.csv", index=False) # another dataset including the Countries and Cities of observations

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\735866760.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: Lak

For each observation, include the first country in which the user made an observation, the country in which the user made their most observations and the percentage of the country of the specific observation of the user's country-portfolio. --> new dataset

In [10]:
df = pd.read_csv("../CWData_clean4.csv")

# 1. for each user, count the number of observations in each country
user_country_counts = (
    df
    .groupby(["created_by", "Country"])
    .size()
    .reset_index(name="n_obs_country")
)

# 2. for each user, find the country in which they did the most observations
max_country_per_user = (
    user_country_counts.loc[user_country_counts.groupby("created_by")["n_obs_country"].idxmax()]
    .rename(columns={"Country": "top_country", "n_obs_country": "top_country_count"})
    .loc[:, ["created_by", "top_country", "top_country_count"]]
)

# 3. total observations per user
total_obs_per_user = (
    df.groupby("created_by").size().reset_index(name="total_obs_user")
)

# 4. first country in which user made an observation
first_country_per_user = (
    df.sort_values("created_at_local")
      .groupby("created_by")["Country"]
      .first()
      .reset_index()
      .rename(columns={"Country": "first_country"})
)

# 5. merge
df = df.merge(max_country_per_user, on="created_by", how="left")
df = df.merge(total_obs_per_user, on="created_by", how="left")
df = df.merge(first_country_per_user, on="created_by", how="left")

# 6. percentage of observations of each user in the country of the current observation
df["percent_in_country"] = (
    df.groupby(["created_by", "Country"])["Country"]
      .transform("count")  # how many observations in this country
      / df["total_obs_user"] * 100
)

df.to_csv("../CWData_clean5.csv",index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\2169036213.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: La

Include first order regions (cantons, states, etc.) in the dataset.

In [12]:
df = pd.read_csv("../CWData_clean5.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["date"] = df["created_at_local"].dt.floor("D")
df = df.dropna(subset=["latitude","longitude"])

regions = gpd.read_file("../Borders/ne_10m_admin_1_states_provinces/ne_10m_admin_1_states_provinces.shp")[["name", "geometry"]]

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

gdf = gdf.drop(columns=["index_right"], errors="ignore")
gdf = gpd.sjoin(gdf, regions, how="left", predicate="within")
gdf = gdf.rename(columns={"name": "Region"})

# spots without any region
regions_proj = regions.to_crs("EPSG:3857")
missing_region = gdf["Region"].isna()
gdf_missing_r = gdf[missing_region].drop(columns=["index_right"], errors="ignore").to_crs("EPSG:3857")
gdf_missing_r = gpd.sjoin_nearest(gdf_missing_r, regions_proj, how="left")
gdf_missing_r = gdf_missing_r[~gdf_missing_r.index.duplicated(keep="first")]
gdf.loc[missing_region, "Region"] = gdf_missing_r["name"].values

gdf.to_csv("../CWData_clean6.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\126671535.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: Lak

In [13]:
df = pd.read_csv("../CWData_clean6.csv")

pd.set_option("display.max_columns", None)
df.tail()

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\3842641729.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: La

,id,root_id,topic_id,topic_role,category,description,image,latitude,longitude,geo_hash,event_id,like_count,like_state,comment_count,flag_count,flag_state,check_sum,check_state,blocking_state,is_featured,is_checked,privacy_type,source_id,source_data,state,spotted_at,spotted_by,spotted_by_type,spotted_by_name,spotted_by_image,spotted_by_topic_role,spotted_by_blocking_state,created_at,created_by,modified_at,modified_by,cluster,quantity,perspective_id,Category,Category_Nr,Waterlevel_Virtual,SoilMoisture,SoilMoisture_Nr,TempStream,TempStream_Nr,Plastic_Amount,Plastic_Amount_Nr,Stream_Width,Stream_Depth,Streambed_Material,Streambed_Material_Nr,FlowVelocity_Method,FlowVelocity_Method_Nr,FlowVelocity_direct_vel,FlowVelocity_PS_dist,FlowVelocity_PS_time1,FlowVelocity_PS_time2,FlowVelocity_PS_time3,Plastic_ObservationTime,Plastic_Location,Plastic_Location_Nr,Plastic_RiverWidth,Plastic_RiverWidth_Nr,Plastic_PET,Plastic_PET_Nr,Plastic_POSoft,Plastic_POSoft_Nr,Plastic_POHard,Plastic_POHard_Nr,Plastic_PS,Plastic_PS_Nr,Plastic_PSE,Plastic_PSE_Nr,Plastic_PMultilayer,Plastic_PMultilayer_Nr,Plastic_POther,Plastic_POther_Nr,Plastic_Shore_Plotsize,Plastic_Shore_Plotsize_Nr,Plastic_Removed,Plastic_River_Stagnant,Waterlevel_Physical_Unit,Waterlevel_Physical_Unit_Nr,Waterlevel_Physical,Streamtype,Streamtype_Nr,Swimming_Quality,Drinking_Quality,Naturality,Naturality_Nr,StreamColor,StreamColor_Nr,Stream_Ground_Visibility,Stream_Ground_Visibility_Nr,Stream_Animals,Stream_Pollution_Reason,Stream_sometimes_dry,Stream_Name,TempStream_snow_ice,Stream_Waterquality,Stream_Waterquality_Nr,Stream_Waterclarity,Stream_Waterclarity_Nr,StreamColor_other,Stream_Vegetation,Stream_Foam,Stream_Foam_Nr,Stream_Algae,Stream_Algae_Nr,Stream_Odor,Stream_Odor_Type,Stream_Odor_Type_Nr,Stream_Odor_Type_other,Stream_Litter,Stream_Litter_Nr,Stream_Flow_Alteration,Stream_Flow_Alteration_Nr,Stream_typical_Color,Stream_typical_Color_Nr,Stream_Drainage_Basin,Stream_Drainage_Basin_Nr,Watertype,Watertype_Nr,Lake_Usage,Lake_Usage_Nr,Lake_Access,Lake_Shore_State,Lake_Shore_State_Nr,Lake_Swimming,Lake_Transparency,Lake_Transparency_Nr,Lake_Color,Lake_Color_Nr,Lake_Odor,Lake_Odor_Nr,Lake_Shore_Vegetation,Lake_Shore_Vegetation_Nr,Lake_Underwater_Vegetation,Lake_Floating_Leaveplants,Lake_Duckweed,Lake_Duckweed_Nr,Lake_Mussels,Lake_Mussels_Nr,Lake_Deadwood,Lake_Deadwood_Nr,Lake_Animals,Lake_Animals_Nr,Lake_Waterlevel_Changes,Lake_Waterlevel_Changes_Nr,Lake_Dry,Lake_Dry_Nr,Image_Gallery,tz_name,created_at_local,geometry,ISO_A3,Country,City,year_month,top_country,top_country_count,total_obs_user,first_country,percent_in_country,date,index_right,Region
68581,1307627,1240218,7,registered,3204,Looking upstream,000007/2026/06/30/nckf2th1dzqiyg6st6hdoq51311h...,51.660360,-1.984260,gcntfu7p7mhd,NaN,0,NaN,0,0,NaN,0,NaN,NaN,0,0,0,NaN,NaN,1,2026-06-29 15:16:58,122129,1,ITAG Karen Shaw,2026/02/28/hukig64tf89lbvdqc1yxmmmjcww2cn0t,registered,NaN,2026-06-30 18:08:44,122129,NaN,NaN,NaN,1,NaN,stream type,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gravel,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fluss,3.0,Yes,No,Natural,1.0,Clear,1.0,Yes,1.0,No,NaN,Yes,River Thames,NaN,Good,2.0,Clear,1.0,NaN,Yes,No,1.0,Growth visible,2.0,No,NaN,NaN,NaN,No,1.0,"No withdrawals, discharge or diversions from t...",1.0,Yes,1.0,NaN,NaN,River and Land,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2243123.0,Europe/London,2026-06-30 19:08:44,POINT (-1.98426 51.66036),GBR,United Kingdom,Bath,2026-06,United Kingdom,57,57,United Kingdom,100.000000,2026-06-30,1704.0,Gloucestershire
68582,1307630,1178707,7,registered,3204,NaN,000007/2026/06/30/lslm3btl9ht18l48yy9ch81dvvzb...,46.663994,10.210327,u0r32qfvmygw,NaN,0,NaN,0,0,NaN,0,NaN,NaN,1,0,0,NaN,NaN,1,2026-06-30 20:21:29,65505,1,Rieke Goe,2022/05/20/3bc6c23c0411346347b34cc4e17acd6b,registered,NaN,2026-06-30 18:22:40,65505,NaN,NaN,NaN

In [14]:
df = pd.read_csv("../CWData_clean6.csv")

df = df[["id","root_id","category","description",
         "latitude","longitude","spotted_at","spotted_by",
         "spotted_by_type","spotted_by_name",
         "spotted_by_topic_role","created_at",
         "created_at_local","created_by","modified_at",
         "modified_by","date","year_month","Country","Region","City",
         "top_country","top_country_count","total_obs_user",
         "first_country","percent_in_country",
         "Category","Category_Nr","Waterlevel_Virtual",
         "SoilMoisture","SoilMoisture_Nr","TempStream",
         "TempStream_Nr","Plastic_Amount","Plastic_Amount_Nr",
         "Stream_Width","Stream_Depth","Streambed_Material",
         "Streambed_Material_Nr","FlowVelocity_Method","FlowVelocity_Method_Nr",
         "FlowVelocity_direct_vel","FlowVelocity_PS_dist","FlowVelocity_PS_time1",
         "FlowVelocity_PS_time2","FlowVelocity_PS_time3","Plastic_ObservationTime",
         "Plastic_Location","Plastic_Location_Nr","Plastic_RiverWidth",
         "Plastic_RiverWidth_Nr","Plastic_PET","Plastic_PET_Nr",
         "Plastic_POSoft","Plastic_POSoft_Nr","Plastic_POHard",
         "Plastic_POHard_Nr","Plastic_PS","Plastic_PS_Nr",
         "Plastic_PSE","Plastic_PSE_Nr","Plastic_PMultilayer",
         "Plastic_PMultilayer_Nr","Plastic_POther","Plastic_POther_Nr",
         "Plastic_Shore_Plotsize","Plastic_Shore_Plotsize_Nr","Plastic_Removed",
         "Plastic_River_Stagnant","Waterlevel_Physical_Unit","Waterlevel_Physical_Unit_Nr",
         "Waterlevel_Physical","Streamtype","Streamtype_Nr",
         "Swimming_Quality","Drinking_Quality","Naturality",
         "Naturality_Nr","StreamColor","StreamColor_Nr",
         "Stream_Ground_Visibility","Stream_Ground_Visibility_Nr","Stream_Animals",
         "Stream_Pollution_Reason","Stream_sometimes_dry","Stream_Name",
         "TempStream_snow_ice","Stream_Waterquality","Stream_Waterquality_Nr",
         "Stream_Waterclarity","Stream_Waterclarity_Nr","StreamColor_other",
         "Stream_Vegetation","Stream_Foam","Stream_Foam_Nr",
         "Stream_Algae","Stream_Algae_Nr","Stream_Odor",
         "Stream_Odor_Type","Stream_Odor_Type_Nr","Stream_Odor_Type_other",
         "Stream_Litter","Stream_Litter_Nr","Stream_Flow_Alteration",
         "Stream_Flow_Alteration_Nr","Stream_typical_Color","Stream_typical_Color_Nr",
         "Stream_Drainage_Basin","Stream_Drainage_Basin_Nr","Watertype",
         "Watertype_Nr","Lake_Usage","Lake_Usage_Nr",
         "Lake_Access","Lake_Shore_State","Lake_Shore_State_Nr",
         "Lake_Swimming","Lake_Transparency","Lake_Transparency_Nr",
         "Lake_Color","Lake_Color_Nr","Lake_Odor",
         "Lake_Odor_Nr","Lake_Shore_Vegetation","Lake_Shore_Vegetation_Nr",
         "Lake_Underwater_Vegetation","Lake_Floating_Leaveplants","Lake_Duckweed",
         "Lake_Duckweed_Nr","Lake_Mussels","Lake_Mussels_Nr",
         "Lake_Deadwood","Lake_Deadwood_Nr","Lake_Animals",
         "Lake_Animals_Nr","Lake_Waterlevel_Changes","Lake_Waterlevel_Changes_Nr",
         "Lake_Dry","Lake_Dry_Nr","Image_Gallery","image","geo_hash"
         ]]

df.to_csv("../CWData_clean7.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\2781630501.py:1: DtypeWarning: Columns (0: topic_role, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Plastic_River_Stagnant, 14: Waterlevel_Physical_Unit, 15: Streamtype, 16: Swimming_Quality, 17: Drinking_Quality, 18: Naturality, 19: StreamColor, 20: Stream_Ground_Visibility, 21: Stream_Animals, 22: Stream_Pollution_Reason, 23: Stream_sometimes_dry, 24: Stream_Name, 25: TempStream_snow_ice, 26: Stream_Waterquality, 27: Stream_Waterclarity, 28: StreamColor_other, 29: Stream_Vegetation, 30: Stream_Foam, 31: Stream_Algae, 32: Stream_Odor, 33: Stream_Odor_Type, 34: Stream_Odor_Type_other, 35: Stream_Litter, 36: Stream_Flow_Alteration, 37: Stream_typical_Color, 38: Stream_Drainage_Basin, 39: Lake_Usage, 40: Lake_Usage_Nr, 41: Lake_Access, 42: La

Create a Switzerland-only dataset

In [18]:
df = pd.read_csv("../CWData_clean7.csv")
df_ch = df[df["Country"] == "Switzerland"]
df_ch.to_csv("../CWData_Switzerland.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\221422150.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

Split dataset into superuser (1000+ obs), one-timers (1 obs), and between (2-999 obs)

In [19]:
df = pd.read_csv("../CWData_clean7.csv")
user_counts = df.groupby("created_by").size()

superuser_ids = user_counts[user_counts >= 1000].index
onetimer_ids = user_counts[user_counts == 1].index
between_ids = user_counts[(user_counts >= 2) & (user_counts <= 999)].index

df_superusers = df[df["created_by"].isin(superuser_ids)]
df_onetimers = df[df["created_by"].isin(onetimer_ids)]
df_between = df[df["created_by"].isin(between_ids)]

df_superusers.to_csv("../CWData_superusers.csv", index=False)
df_onetimers.to_csv("../CWData_onetimers.csv", index=False)
df_between.to_csv("../CWData_between.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_28708\219153603.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4